In [1]:
# 07_robustness.ipynb
# Identification robustness (learner family x lever benchmark), the pre-registered
# split-sample sign screen, and the cross-national sign test.
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, pyreadstat
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GroupKFold
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"axes.edgecolor":"0.15","grid.color":"0.85","font.size":10,"axes.titlesize":1})
BASE = Path("..") if (Path("../data")).exists() else Path(".")
DATA, FIG, TAB = BASE/"data", BASE/"results/figures", BASE/"results/tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def savefig(fig, name):
    for ext in ("png","pdf"): fig.savefig(FIG/(name+"."+ext), dpi=600, bbox_inches="tight")
    plt.close(fig)
PVS=[f"PV{i}CIL" for i in range(1,6)]
rng=np.random.default_rng(0)
def wR2(y,yh,w):
    yb=np.average(y,weights=w); return 1-np.sum(w*(y-yh)**2)/np.sum(w*(y-yb)**2)
def wpct(v,wt,p):
    m=~np.isnan(v); vs=np.sort(v[m]); ww=wt[m][np.argsort(v[m])]; return np.interp(p,np.cumsum(ww)/ww.sum(),vs)

from scipy.stats import kendalltau
from math import comb
stu,_=pyreadstat.read_sas7bdat(str(DATA/"bsgkori3.sas7bdat"))
USED=["NISB","S_HOMLIT","S_INTNET","S_HISEI","S_EXCOMP","S_ICTSTUD","S_LRNINTS","S_LRNINTO","S_ACMULT",
      "S_GENEFF","S_SPECEFF","S_CTCLS","S_LRNSAFE","S_GENCLASS","S_SPECLASS","S_ICTPOSG","S_ICTFUT","S_ICTPOSS"]
for c in USED: stu.loc[(stu[c]>=995)|(stu[c]<=-995),c]=np.nan
stu.loc[(stu.NISB<-6)|(stu.NISB>6),"NISB"]=np.nan
stu["CIL5"]=stu[PVS].mean(1)
w=stu.TOTWGTS.to_numpy(); x=stu.NISB.to_numpy(); ok=~np.isnan(x)
o=np.argsort(x[ok]); cw=np.cumsum(w[ok][o])/w[ok].sum(); q=np.interp([1/3,2/3],cw,x[ok][o])
stu["SES3"]=pd.Series(np.where(x<=q[0],"Low",np.where(x<=q[1],"Mid","High"))).where(ok).values
stu["SEX"]=stu.S_SEX.map({0.0:"Sex0",1.0:"Sex1"})
df=stu.dropna(subset=["SES3","SEX"]).copy().reset_index(drop=True); df["grp"]=df.SES3+"_"+df.SEX
LEVERS=["S_LRNINTO","S_ICTPOSS","S_GENCLASS","S_LRNSAFE"]
CTX=["NISB","S_HISEI","S_HOMLIT","S_INTNET","S_EXCOMP","S_GENEFF","S_SPECEFF","S_ICTPOSG","S_ICTFUT","S_ACMULT","S_LRNINTS","S_CTCLS","S_SPECLASS","S_ICTSTUD"]
def ctor(name):
    if name=="HGB": return HistGradientBoostingRegressor(max_iter=400,learning_rate=0.05,max_leaf_nodes=31,l2_regularization=1.0,random_state=0)
    if name=="RF": return RandomForestRegressor(n_estimators=300,n_jobs=-1,random_state=0)
    if name=="ET": return ExtraTreesRegressor(n_estimators=300,n_jobs=-1,random_state=0)
    if name=="EN": return ElasticNet(alpha=0.1,random_state=0)
def order_for(levers,name="HGB",bench=0.75):
    feat=levers+[c for c in CTX if c not in levers]; W=df.TOTWGTS.to_numpy()
    Xb=df[feat].to_numpy(float); Xc=Xb.copy()
    for j,L in enumerate(levers):
        t=wpct(Xb[:,j],W,bench); col=Xc[:,j]; Xc[:,j]=np.where(np.isnan(col),col,np.maximum(col,t))
    if name in ("RF","ET","EN"):
        med=np.nanmedian(Xb,axis=0); Xb=np.where(np.isnan(Xb),med,Xb); Xc=np.where(np.isnan(Xc),med,Xc)
    dd=np.zeros((len(df),5))
    for k,pv in enumerate(PVS):
        m=ctor(name).fit(Xb,df[pv].to_numpy(float),sample_weight=(W if name!="EN" else None)); dd[:,k]=m.predict(Xc)-m.predict(Xb)
    t=df.assign(dCIL=dd.mean(1)).groupby("grp").apply(lambda z:np.average(z.dCIL,weights=z.TOTWGTS))
    return list(t.sort_values(ascending=False).index)
base_order=order_for(LEVERS,"HGB",0.75); rows=[]
for name in ["HGB","RF","ET","EN"]:
    for bench in [0.60,0.75,0.90]:
        od=order_for(LEVERS,name,bench); tau=kendalltau([base_order.index(xx) for xx in od],list(range(len(od)))).correlation
        rows.append(dict(estimator=name,benchmark=bench,tau_vs_base=round(tau,2),order=" > ".join(od)))
rob=pd.DataFrame(rows); rob.to_csv(TAB/"rev_identif_robust.csv",index=False)
print("IDENTIFICATION ROBUSTNESS (Kendall tau vs base order):"); print(rob[["estimator","benchmark","tau_vs_base"]].to_string(index=False))
print("lowest-priority group in every config:", set(od.split(" > ")[-1] for od in rob.order))
# ---- pre-registered split-sample sign screen ----
schs=df.IDSCHOOL.unique(); half=rng.choice(schs,len(schs)//2,replace=False); mask=df.IDSCHOOL.isin(half).to_numpy()
dA=df[mask]; dB=df[~mask]
def wcorr(a,b,wt):
    m=~np.isnan(a); a,b,wt=a[m],b[m],wt[m]; am=np.average(a,weights=wt); bm=np.average(b,weights=wt)
    return np.average((a-am)*(b-bm),weights=wt)/np.sqrt(np.average((a-am)**2,weights=wt)*np.average((b-bm)**2,weights=wt))
cand=["S_LRNINTO","S_ICTPOSS","S_GENCLASS","S_LRNSAFE","S_SPECLASS","S_CTCLS","S_LRNINTS"]
kept=[c for c in cand if wcorr(dA[c].to_numpy(),dA.CIL5.to_numpy(),dA.TOTWGTS.to_numpy())>0]
globals()["df"]=dB  # estimate held-out order with signs fixed on split A
odB=order_for(kept,"HGB",0.75); globals()["df"]=stu.dropna(subset=["SES3","SEX"]).copy().reset_index(drop=True); df["grp"]=df.SES3+"_"+df.SEX
print("\nSPLIT-SAMPLE sign screen: levers kept on A =",kept); print("held-out order on B:"," > ".join(odB))
# ---- cross-national sign test ----
cc=pd.read_csv(TAB/"cross_country.csv"); neg=int((cc.corr_base_sens<0).sum()); n=len(cc)
p=sum(comb(n,k) for k in range(neg,n+1))/2**n
print(f"\nCROSS-NATIONAL sign test: {neg}/{n} negative, one-sided binomial p={p:.4f}")


IDENTIFICATION ROBUSTNESS (Kendall tau vs base order):
estimator  benchmark  tau_vs_base
      HGB       0.60         1.00
      HGB       0.75         1.00
      HGB       0.90         0.87
       RF       0.60         0.60
       RF       0.75         0.47
       RF       0.90         0.60
       ET       0.60         0.73
       ET       0.75         0.47
       ET       0.90         0.60
       EN       0.60         0.87
       EN       0.75         0.47
       EN       0.90         0.33
lowest-priority group in every config: {'High_Sex1', 'High_Sex0'}



SPLIT-SAMPLE sign screen: levers kept on A = ['S_LRNINTO', 'S_ICTPOSS', 'S_LRNSAFE']
held-out order on B: Low_Sex1 > Low_Sex0 > Mid_Sex0 > Mid_Sex1 > High_Sex0 > High_Sex1

CROSS-NATIONAL sign test: 7/8 negative, one-sided binomial p=0.0352
